<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/01_explorar_tus_textos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 01 · Explora tus textos

**Objetivo:** cargar un corpus (colección de textos) y conocerlo un poco antes de entrenar.

Un modelo de lenguaje solo puede aprender lo que hay en sus datos. Por eso, en Humanidades Digitales
lo primero es siempre **mirar el corpus**: ¿cuánto texto hay?, ¿qué palabras aparecen?, ¿hay «ruido»
(números de página, cabeceras, errores de OCR)?

Usaremos como ejemplo la *Primera parte del Quijote* y el *Lazarillo de Tormes* (dominio público,
Project Gutenberg). Puedes usar tus propios textos `.txt`.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes
Cambia aquí la carpeta con tus textos. `data/sample` contiene los textos de ejemplo.

In [ ]:
CARPETAS = ["data/sample"]     # p. ej. ["data/mis_textos"]
PALABRA_CLAVE = "Dulcinea"     # para buscar en contexto (KWIC)

### (Opcional) Sube tus propios textos
En Colab, ejecuta la celda siguiente con `SUBIR = True` para elegir archivos `.txt` de tu ordenador.
Se guardarán en `data/mis_textos/`. Después cambia `CARPETAS = ["data/mis_textos"]` arriba.

In [ ]:
SUBIR = False
if SUBIR:
    from dhllm.data import upload_texts
    upload_texts("data/mis_textos")

## 1. Cargar los textos

In [ ]:
from dhllm.data import load_texts
textos = load_texts(CARPETAS)
textos

Veamos el principio del primer texto:

In [ ]:
print(textos[0].content[:800])

## 2. Estadísticas básicas
- **riqueza léxica** = palabras distintas ÷ palabras totales (*type-token ratio*). Depende mucho de la longitud del texto.

In [ ]:
from dhllm.data import corpus_stats
tabla = corpus_stats(textos)

## 3. Palabras más frecuentes
Quitamos las *palabras vacías* (de, la, que, y...) para ver las palabras con contenido.

In [ ]:
from dhllm.data import top_words
top_words(textos, n=25);

## 4. Palabra clave en contexto (KWIC)
Una herramienta clásica de la lingüística de corpus.

In [ ]:
from dhllm.data import kwic
kwic(textos, PALABRA_CLAVE, width=45, max_results=8);

## 5. ¿Cuánto texto necesita un modelo?
Para comparar: GPT-3 se entrenó con unos 300 000 millones de tokens; los modelos actuales con más de
10 billones. Nuestro corpus de ejemplo tiene ~1 millón de caracteres. Por eso nuestros modelos serán
**miniaturas**: aprenderán algo del estilo, pero no «sabrán» cosas.

In [ ]:
from dhllm.data import join_texts
corpus = join_texts(textos)
print(f"Caracteres: {len(corpus):,}".replace(",", "."))
print(f"Palabras aprox.: {len(corpus.split()):,}".replace(",", "."))

## 🖐️ Tu turno
1. Busca otras palabras con `kwic` (por ejemplo *molinos*, *honra*, *pan*). ¿Qué te dice el contexto?
2. Cambia `n=25` por `n=50` en `top_words`. ¿Aparecen nombres propios?
3. Sube un texto propio y compara sus estadísticas con las del Quijote.
4. **Para pensar:** ¿qué «ruido» ves en tus textos que un modelo aprendería a imitar?